# OpenPlaque — Blind Source-CCTA Coronary Ostium Discovery v1.2 Fast
Performance-safe continuation of v1.1. Scientific thresholds, discovery weights, RCA control criteria, and the 40-component limit are unchanged. v1.2 adds resumable vesselness caching, progress reporting, and skips serial 9-plane QC only for beam finalists that already fail a non-QC acceptance term and therefore cannot be accepted.

Designed for **Runtime → Run all**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/OpenPlaque'
OUTPUT_DIR = DRIVE_ROOT + '/Left_Coronary_Source_Ostium_Discovery_v1_2_fast'
VESSELNESS_CACHE_DIR = DRIVE_ROOT + '/Cache/Left_Coronary_Source_Ostium_Discovery_v1_2'
REUSE_VESSELNESS_CACHE = True
BRANCH = 'left-coronary-source-ostium-discovery-from-main'
PINNED_SCIENCE_COMMIT = '205d38d3c1a3c07158b53e347f4e4690bd5d219e'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
EXPECTED_ALGORITHM = 'left-coronary-source-ostium-discovery-v1.2-performance'
print('Branch:', BRANCH)
print('Pinned science commit:', PINNED_SCIENCE_COMMIT)
print('Output:', OUTPUT_DIR)
print('Vesselness cache:', VESSELNESS_CACHE_DIR)
print('Reuse vesselness cache:', REUSE_VESSELNESS_CACHE)


In [ ]:
import os, shutil
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
!git clone --depth 30 --branch $BRANCH https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout --detach $PINNED_SCIENCE_COMMIT
HEAD = !git -C /content/OpenPlaque rev-parse HEAD
HEAD = HEAD[0].strip()
print('Checked out HEAD:', HEAD)
assert HEAD == PINNED_SCIENCE_COMMIT, (HEAD, PINNED_SCIENCE_COMMIT)


In [ ]:
%pip uninstall -y openplaque >/dev/null 2>&1
%pip install -q --no-cache-dir --force-reinstall --no-deps /content/OpenPlaque
%pip install -q pytest SimpleITK scipy pandas matplotlib numpy pydicom psutil


In [ ]:
import sys, importlib, pathlib, pytest
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
importlib.invalidate_caches()
import openplaque
from openplaque import left_coronary_source_ostium_discovery_v1_2 as exp
print('openplaque:', openplaque.__file__)
print('experiment:', exp.__file__)
print('algorithm:', exp.ALGORITHM)
assert exp.BASELINE == BASELINE
assert exp.ALGORITHM == EXPECTED_ALGORITHM
src_text = pathlib.Path(exp.__file__).read_text()
compile(src_text, exp.__file__, 'exec')
assert '_cheap_path_gate' in src_text
assert 'root_vesselness_partial.npy' in src_text
print('synthetic self-test:', exp.synthetic_root_component_self_test())
test_file='/content/OpenPlaque/tests/test_left_coronary_source_ostium_discovery_v1_2.py'
test_text=pathlib.Path(test_file).read_text()
assert 'left_coronary_source_ostium_discovery_v1_2 as m' in test_text
rc = pytest.main(['-q', test_file])
if rc != 0: raise RuntimeError(f'pytest failed with exit code {rc}')


In [ ]:
from pathlib import Path
root=Path(DRIVE_ROOT)
required=[
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
 root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
 root/'Joint_Three_Vessel_Template_Classifier_v1/candidate_04_source_path.csv',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/aorta.nii.gz',
]
missing=[str(p) for p in required if not p.exists()]
print('Preflight required:',len(required),'missing:',len(missing))
if missing: raise FileNotFoundError('\n'.join(missing))


## Scientific run
The cell prints and writes `progress.json` at each vesselness scale and component. If interrupted after a vesselness scale completes, rerunning the notebook resumes from the saved scale cache instead of recomputing it.


In [ ]:
import gc
from openplaque.left_coronary_source_ostium_discovery_v1_2 import run
gc.collect()
result = run(DRIVE_ROOT, OUTPUT_DIR, reuse_vesselness_cache=REUSE_VESSELNESS_CACHE)
s=result['summary']
print('STATUS:', s['status'])
print('BLIND ROOT COMPONENTS:', s.get('n_blind_root_components'))
print('TRACED COMPONENTS:', s.get('n_traced_components'))
print('RCA CONTROL PASS:', s.get('RCA_blind_control',{}).get('control_pass'))
print('SECOND CORONARY:', s.get('second_coronary_candidate'))
print('REPORT:', result.get('report'))
print('ZIP:', result.get('zip'))
